In [3]:
# SMOTE on the Titanic data: balance survivors / non-survivors in the TRAINING data
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

df = pd.read_csv('E:/Vishal_Adesh/Data Analytics/Day5/titanic.csv')
# Only drop rows missing the columns we use (dropna() on the whole table would also
# drop every passenger without a 'deck' value - almost 80% of the data)
df = df.dropna(subset=['age', 'pclass', 'fare'])
X = df[['age', 'pclass', 'fare']]
y = df['survived']
print("Rows:", len(df))
print("Class distribution:\n", y.value_counts())

# Split first, then apply SMOTE to the training part only
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)
print("\nTraining rows before SMOTE:", len(X_train), dict(y_train.value_counts()))
print("Training rows after SMOTE :", len(X_res), dict(pd.Series(y_res).value_counts()))

# The synthetic rows are appended after the original ones
synthetic = X_res.iloc[len(X_train):].assign(survived=y_res.iloc[len(X_train):].values)
print("\nFirst 5 synthetic records (ages and fares are new in-between values):")
print(synthetic.head().round(2))

# Does it help? Same model, with and without SMOTE, evaluated on the untouched test set
for name, (Xt, yt) in {'Without SMOTE': (X_train, y_train), 'With SMOTE': (X_res, y_res)}.items():
    model = LogisticRegression(max_iter=1000).fit(Xt, yt)
    print(f"\n{name}")
    print(classification_report(y_test, model.predict(X_test), digits=3))
# SMOTE usually raises recall for the minority class (survivors) at the cost of some precision


Rows: 714
Class distribution:
 survived
0    424
1    290
Name: count, dtype: int64

Training rows before SMOTE: 535 {0: 318, 1: 217}
Training rows after SMOTE : 636 {1: 318, 0: 318}

First 5 synthetic records (ages and fares are new in-between values):
       age  pclass    fare  survived
535  24.00       1   69.30         1
536  16.19       1   30.04         1
537  25.23       1   30.00         1
538  22.01       1  262.69         1
539  48.00       1   26.25         1

Without SMOTE
              precision    recall  f1-score   support

           0      0.706     0.840     0.767       106
           1      0.679     0.493     0.571        73

    accuracy                          0.698       179
   macro avg      0.693     0.666     0.669       179
weighted avg      0.695     0.698     0.687       179


With SMOTE
              precision    recall  f1-score   support

           0      0.724     0.717     0.720       106
           1      0.595     0.603     0.599        73

    ac